# 06 · Regional comparison: Benguela vs Agulhas
The study area is split at Cape Agulhas (20°E): **Benguela** (16–20°E, Atlantic, upwelling-dominated) and **Agulhas** (20–34°E, Indian Ocean, warm western boundary current). Input: `scripts/build_subregion_series.py`. Each series is a cos-latitude area-weighted mean of surface (0.494 m) ocean cells, so it includes offshore water as well as the coastal upwelling band.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline
from src import modelling
import pandas as pd, numpy as np, matplotlib.pyplot as plt
visualisation.apply_style()
df = pd.read_csv('../data/processed/thetao_subregions_monthly.csv', index_col=0, parse_dates=True).asfreq('MS')
names = {k: v['name'] for k, v in settings['sub_regions'].items()}
COLORS = {'benguela': '#2b6cb0', 'agulhas': '#c53030'}

## Where the regions are

In [ ]:
import xarray as xr
clim = xr.open_dataset('../data/processed/thetao_monthly_climatology.nc')['thetao_climatology'].mean('month')
fig, ax = visualisation.plot_map(clim, 'Mean surface temperature 1993–2025 and sub-regions')
ax.axvline(20.0, color='k', ls='--', lw=1)
ax.text(18.0, -28.4, 'Benguela', ha='center', fontsize=9)
ax.text(27.0, -28.4, 'Agulhas', ha='center', fontsize=9)
visualisation.save(fig, '06_region_map')

## Descriptive statistics

In [ ]:
stats = pd.concat({k: preprocessing.describe_series(df[k])['value'] for k in df}, axis=1)
stats.to_csv('../outputs/tables/06_region_stats.csv')
stats

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for k in df:
    ax.plot(df.index, df[k], lw=1, color=COLORS[k], label=names[k])
ax.set(title='Area-weighted surface temperature by region', ylabel='Temperature (°C)')
ax.set_ylim(top=df.max().max() + 1.8)
ax.legend(loc='upper left', ncol=2, fontsize=8); visualisation.source_note(ax); fig.tight_layout()
visualisation.save(fig, '06_region_timeseries')

## Seasonal cycle

In [ ]:
clims = pd.DataFrame({k: preprocessing.monthly_climatology(df[k]) for k in df})
fig, ax = plt.subplots(figsize=(5.5, 3))
for k in clims:
    ax.plot(clims.index, clims[k], marker='o', color=COLORS[k], label=names[k])
ax.set(title='Monthly climatology 1993–2025', xlabel='Month', ylabel='Temperature (°C)', xticks=range(1, 13))
ax.legend(fontsize=8); visualisation.source_note(ax); fig.tight_layout()
visualisation.save(fig, '06_region_seasonality')
amp = (clims.max() - clims.min()).rename('seasonal_amplitude_degC')
peak = clims.idxmax().rename('warmest_month'); low = clims.idxmin().rename('coolest_month')
pd.concat([amp, peak, low], axis=1).round(2)

## Anomalies and trends
HAC (Newey–West) standard errors account for autocorrelation.

In [ ]:
anoms = pd.DataFrame({k: preprocessing.anomalies(df[k]) for k in df})
trends = pd.DataFrame({k: preprocessing.hac_trend(anoms[k]) for k in anoms}).T
trends.to_csv('../outputs/tables/06_region_trends.csv')
trends.round(4)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
for ax, k in zip(axes, anoms):
    a = anoms[k]; ax.fill_between(a.index, 0, a, where=a > 0, color='#c53030', lw=0)
    ax.fill_between(a.index, 0, a, where=a < 0, color='#2b6cb0', lw=0)
    ax.plot(a.rolling(12, center=True).mean(), color='k', lw=1)
    t = trends.loc[k]
    ax.set_title(f"{names[k]}: {t.trend_degC_per_decade:+.2f} ± {t.ci95:.2f} °C/decade", fontsize=9)
    ax.set_ylabel('Anomaly (°C)')
visualisation.source_note(axes[-1]); fig.tight_layout()
visualisation.save(fig, '06_region_anomalies')

In [ ]:
r = anoms.corr().iloc[0, 1]
r12 = anoms.rolling(12).mean().corr().iloc[0, 1]
print(f'Correlation of monthly anomalies: {r:.2f}; of 12-month running means: {r12:.2f}')

## One-month-ahead forecast skill by region (validation 2016–2019; test period untouched)

In [ ]:
m = settings['modelling']; rows = {}
for k in df:
    sp = modelling.chronological_split(modelling.make_features(df[k], m['lags']),
                                       m['train_end'], m['validation_end'], m['test_end'])
    rows[k] = modelling.evaluate(sp, settings['project']['random_seed'], on='validation')
skill = pd.concat(rows, names=['region', 'model'])
skill.round(3).to_csv('../outputs/tables/06_region_validation_metrics.csv')
skill[['MAE', 'RMSE', 'bias', 'skill_vs_climatology']].round(3)